In [182]:
import json
import re

import requests
from bs4 import BeautifulSoup

In [ ]:
url = 'https://www.youtube.com/watch?v=K5KVEU3aaeQ'
headers = {'User-Agent': 'Mozilla/5.0'}

html = requests.get(url, headers=headers).text
soup = BeautifulSoup(html, 'html.parser')


def extract_json(key):
    for script in soup.find_all('script'):
        if key in script.text:
            match = re.search(rf'{key}\s*=\s*({{.*}});', script.text)
            if match:
                try:
                    return json.loads(match.group(1))
                except Exception as e:
                    print(f'Error occurred while parsing {key}: {e}')
    return None


initial_data = extract_json('ytInitialData')
player_data = extract_json('ytInitialPlayerResponse')

chapters = []

try:
    markers = player_data['playerOverlays']['playerOverlayRenderer']['decoratedPlayerBarRenderer'][
        'decoratedPlayerBarRenderer'
    ]['playerBar']['multiMarkersPlayerBarRenderer']['markersMap']

    for item in markers:
        if item['key'] == 'AUTO_CHAPTERS':
            for ch in item['value']['chapters']:
                chapters.append(
                    {
                        'title': ch['chapterRenderer']['title']['simpleText'],
                        'start_seconds': ch['chapterRenderer']['timeRangeStartMillis'] // 1000,
                    }
                )

except (KeyError, TypeError):
    pass

if not chapters:
    try:
        markers = initial_data['playerOverlays']['playerOverlayRenderer']['decoratedPlayerBarRenderer'][
            'decoratedPlayerBarRenderer'
        ]['playerBar']['multiMarkersPlayerBarRenderer']['markersMap']

        # find AUTO_CHAPTERS
        chapters_data = None
        for item in markers:
            if item['key'] == 'AUTO_CHAPTERS':
                chapters_data = item['value']['chapters']

        for ch in chapters_data:
            title = ch['chapterRenderer']['title']['simpleText']
            time_ms = ch['chapterRenderer']['timeRangeStartMillis']
            time_sec = time_ms // 1000

            chapters.append({'title': title, 'start_seconds': time_sec})

    except (KeyError, TypeError):
        pass


if not chapters:
    try:
        description = player_data['videoDetails']['shortDescription']

        matches = re.findall(r'(\d{1,2}:\d{2})\s+(.+)', description)

        for time_str, title in matches:
            minutes, seconds = map(int, time_str.split(':'))
            total_seconds = minutes * 60 + seconds

            chapters.append({'title': title.strip(), 'start_seconds': total_seconds})

    except (KeyError, TypeError):
        pass


if chapters:
    print('Chapters found:')
    print(chapters)
else:
    print('No chapters available for this video')

Chapters found:
[{'title': 'Introduction', 'start_seconds': 0}, {'title': 'What is Python?', 'start_seconds': 56}, {'title': 'Installing Python', 'start_seconds': 251}, {'title': 'Python Interpreter', 'start_seconds': 336}, {'title': 'Code Editors', 'start_seconds': 450}, {'title': 'Your First Python Program', 'start_seconds': 529}, {'title': 'Python Extension', 'start_seconds': 745}, {'title': 'Linting Python Code', 'start_seconds': 866}, {'title': 'Formatting Python Code', 'start_seconds': 1120}, {'title': 'Running Python Code', 'start_seconds': 1371}, {'title': 'Python Implementations', 'start_seconds': 1470}, {'title': 'How Python Code is Executed', 'start_seconds': 1619}, {'title': 'Quiz', 'start_seconds': 1785}, {'title': 'Python Mastery Course', 'start_seconds': 1877}, {'title': 'Variables', 'start_seconds': 1904}, {'title': 'Variable Names', 'start_seconds': 2088}, {'title': 'Strings', 'start_seconds': 2271}, {'title': 'Escape Sequences', 'start_seconds': 2600}, {'title': 'Form

In [296]:
len(chapters)

44

In [ ]:
# to be tested
try:
    cards = player_data['cards']['cardCollectionRenderer']['cards']
    print(cards)

    card_items = []
    for card in cards:
        r = card['cardRenderer']
        card_items.append(
            {
                'teaser_text': r.get('teaser', {})
                .get('simpleCardTeaserRenderer', {})
                .get('message', {})
                .get('simpleText', ''),
                'start_ms': r.get('startCardActiveMs'),
            }
        )

    card_count = len(card_items)

except Exception as e:
    print(f'Error occurred while parsing card data: {e}')
    card_count = 0

card_count

[{'cardRenderer': {'teaser': {'simpleCardTeaserRenderer': {'message': {'simpleText': 'الاطّلاع على التصحيحَين'}, 'trackingParams': 'CBIQ0DYiEwiN96mG0teTAxWsWU8EHWI3FqI=', 'prominent': True, 'logVisibilityUpdates': True, 'onTapCommand': {'clickTrackingParams': 'CBIQ0DYiEwiN96mG0teTAxWsWU8EHWI3FqLKAQTStLV8', 'changeEngagementPanelVisibilityAction': {'targetId': 'engagement-panel-error-corrections', 'visibility': 'ENGAGEMENT_PANEL_VISIBILITY_EXPANDED'}}}}, 'cueRanges': [{'startCardActiveMs': '0', 'endCardActiveMs': '5000', 'teaserDurationMs': '6000', 'iconAfterTeaserMs': '5000'}], 'trackingParams': 'CBEQtZcBGAAiEwiN96mG0teTAxWsWU8EHWI3FqI='}}]


1

In [ ]:
# is_embeddable to be tested
try:
    playability = player_data['playabilityStatus']
    status = playability['status']  # OK, LOGIN_REQUIRED, UNPLAYABLE
    is_age_restricted = status == 'LOGIN_REQUIRED'
    is_embeddable = playability.get('playableInEmbed', True)

    # Miniplayer support
    supports_miniplayer = player_data['microformat']['playerMicroformatRenderer'].get('isFamilySafe', False)
except Exception as e:
    print(f'Error occurred while parsing playability data: {e}')
    is_age_restricted = False
    is_embeddable = True

is_age_restricted, is_embeddable

(False, True)

In [ ]:
# ── Verified badge — works for Arabic AND English ────────────
VERIFIED_LABELS = {
    'Verified',
}

VERIFIED_STYLES = {'BADGE_STYLE_TYPE_VERIFIED'}

VERIFIED_ICONS = {'CHECK_CIRCLE_THICK'}

try:
    badges = initial_data['contents']['twoColumnWatchNextResults']['results']['results']['contents'][1][
        'videoSecondaryInfoRenderer'
    ]['owner']['videoOwnerRenderer']['badges']

    is_verified = False
    badge_labels = []

    for b in badges:
        renderer = b.get('metadataBadgeRenderer', {})

        label = renderer.get('accessibilityData', {}).get('label', '')
        style = renderer.get('style', '')
        icon_type = renderer.get('icon', {}).get('iconType', '')
        tooltip = renderer.get('tooltip', '')

        badge_labels.append(label)

        # Check any of the 3 signals
        if label in VERIFIED_LABELS or style in VERIFIED_STYLES or icon_type in VERIFIED_ICONS:
            is_verified = True

    print(f'is_verified  : {is_verified}')
    print(badges)

except (KeyError, TypeError):
    is_verified = False
    badge_labels = []

is_verified  : True
[{'metadataBadgeRenderer': {'icon': {'iconType': 'CHECK_CIRCLE_THICK'}, 'style': 'BADGE_STYLE_TYPE_VERIFIED', 'tooltip': 'تم إثبات الملكية', 'trackingParams': 'CJEEEOE5IhMI0f-phtLXkwMV27w3CB2v2wrz', 'accessibilityData': {'label': 'تم إثبات الملكية'}}}]


In [ ]:
def _extract_comments_disabled(initial_data: dict | None) -> bool:
    try:
        contents = initial_data['contents']['twoColumnWatchNextResults']['results']['results']['contents']
        for item in contents:
            for c in item.get('itemSectionRenderer', {}).get('contents', []):
                if 'messageRenderer' in c:
                    return True  # messageRenderer in comments section = comments disabled
    except (KeyError, TypeError):
        pass
    return False


_extract_comments_disabled(initial_data)

False

In [ ]:
def _extract_paid_promotion(player_data: dict | None) -> bool:
    try:
        return bool(player_data['paidContentOverlay'])
    except (KeyError, TypeError):
        return False


_extract_paid_promotion(player_data)

False

In [281]:
player_data['paidContentOverlay']

{'paidContentOverlayRenderer': {'text': {'simpleText': 'يتضمن إعلانًا ترويجيًا مدفوعًا'},
  'durationMs': '10000',
  'navigationEndpoint': {'clickTrackingParams': 'CAEQwYYIIhMI8_WNi9DXkwMVeYGxAx2BWSyiygEEY05sow==',
   'commandMetadata': {'webCommandMetadata': {'url': 'https://support.google.com/youtube?p=ppp&nohelpkit=1',
     'webPageType': 'WEB_PAGE_TYPE_UNKNOWN',
     'rootVe': 83769}},
   'urlEndpoint': {'url': 'https://support.google.com/youtube?p=ppp&nohelpkit=1',
    'target': 'TARGET_NEW_WINDOW',
    'grwOpenInOverride': 'GRW_OPEN_IN_OVERRIDE_USE_PREFERRED_APP_NO_PROMPT'}},
  'icon': {'iconType': 'MONEY_HAND'},
  'showInPip': True,
  'trackingParams': 'CAEQwYYIIhMI8_WNi9DXkwMVeYGxAx2BWSyi'}}